# 07 — Combined Riparian Hotspot Report

**Goal:** notebook 06 ended on its own conclusion: the 500m within-cell grid scan (06) and the
1km wide-radius comparison (05) are complementary, not redundant — the grid scan finds sharp
local edges (Mathare) with no prior knowledge needed, but structurally misses settlements that
are uniformly dense on both sides of the riverside line (Kibera: +0.7pp at grid grain vs. +8.5pp
at 1km radius). Neither method alone is a complete monitoring tool. This notebook builds the
combined output that was left as the recommended next step: run both signals together and merge
them into a single ranked hotspot report.

**Approach:** run the fast, no-prior-knowledge grid scan first (as in 06) over the whole city.
From it, flag two kinds of candidates instead of one:
- **narrow-edge candidates** — cells with a large riverside-vs-rest-of-cell diff (06's original
  signal, catches sharp local boundaries like Mathare).
- **saturation candidates** — cells where *both* riverside and rest-of-cell are already highly
  built-up (the Kibera failure mode: high absolute riverside built-up with no local contrast to
  find at grid grain).

Only this small merged candidate set (not all 3016 cells) then gets the expensive 1km wide-radius
verification pass from notebook 05, confirming which candidates actually show a wider-context
riverside signal. This keeps the expensive step cheap by only running it where it's needed,
instead of computing it city-wide.

In [1]:
import sys
sys.path.insert(0, '../src')

import ee
import geemap
import urllib.request
from acquisition import get_nairobi_boundary, get_sentinel2_composite
from classification import (
    build_feature_image, get_worldcover_builtup, sample_training_points,
    train_random_forest, classify_builtup,
)

ee.Initialize(project='solar-haven-349708')

nairobi = get_nairobi_boundary()
proj = ee.Projection('EPSG:32737')  # UTM 37S — correct hemisphere for Nairobi
grid = nairobi.coveringGrid(proj, 500)
grid = grid.map(lambda f: f.set('centroid', f.geometry().centroid(1).coordinates()))
print('Grid cells:', grid.size().getInfo())

rivers = ee.FeatureCollection('WWF/HydroSHEDS/v1/FreeFlowingRivers').filterBounds(nairobi)
dist_to_river = rivers.distance(searchRadius=200, maxError=10).clip(nairobi)
riverside_mask = dist_to_river.lte(30)

Grid cells: 3016


## Classify 2024 built-up (same procedure as notebooks 03/05/06)

In [2]:
composite, scene_count = get_sentinel2_composite(
    nairobi, start_date='2024-06-01', end_date='2024-09-30', cloud_threshold=20
)
features = build_feature_image(composite)
worldcover_builtup = get_worldcover_builtup(nairobi)
train_samples, test_samples = sample_training_points(features, worldcover_builtup, nairobi)
classifier = train_random_forest(train_samples)
builtup = classify_builtup(features, classifier)

test_accuracy = test_samples.classify(classifier).errorMatrix(
    'builtup', 'classification'
).accuracy().getInfo()
print(f'Scenes: {scene_count}, held-out accuracy (sanity check): {test_accuracy * 100:.1f}%')

Scenes: 11, held-out accuracy (sanity check): 85.9%


## Pass 1 — city-wide narrow-grid scan (notebook 06's method, unchanged)

Per-cell riverside vs. rest-of-cell built-up fraction, one `reduceRegions` pass per side, dropped
below the same 20-pixel minimum used in notebook 06.

In [3]:
builtup_riverside = builtup.updateMask(riverside_mask).rename('b')
builtup_rest = builtup.updateMask(riverside_mask.Not()).rename('b')

reducer = ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True)

stats_riverside = builtup_riverside.reduceRegions(collection=grid, reducer=reducer, scale=10).getInfo()['features']
stats_rest = builtup_rest.reduceRegions(collection=grid, reducer=reducer, scale=10).getInfo()['features']

MIN_PIXELS = 20
rows = []
for r, s in zip(stats_riverside, stats_rest):
    pr, ps = r['properties'], s['properties']
    if pr['count'] is None or pr['count'] < MIN_PIXELS:
        continue
    if ps['count'] is None or ps['count'] < MIN_PIXELS:
        continue
    lon, lat = pr['centroid']
    rows.append({
        'lon': lon, 'lat': lat,
        'riverside_pct': pr['mean'] * 100, 'riverside_n': pr['count'],
        'rest_pct': ps['mean'] * 100, 'rest_n': ps['count'],
        'diff_pp': (pr['mean'] - ps['mean']) * 100,
    })

print(f'{len(rows)} of {len(stats_riverside)} cells have enough pixels on both sides to compare.')

578 of 3016 cells have enough pixels on both sides to compare.


## Flag two kinds of candidates

`diff_pp` alone (notebook 06's ranking) only catches sharp local edges. Add a second criterion —
both sides already highly built-up — to also catch cells like Kibera's, where the local contrast
is gone but the absolute riverside built-up fraction is still a real signal at wider context.

Thresholds (`TOP_N`, `SATURATION_PCT`) are analytical choices, not validated cutoffs — same
caveat notebook 06 raised about its own grid resolution and pixel-count filter.

In [4]:
TOP_N = 25
SATURATION_PCT = 70  # both sides at/above this -> no local edge left for the narrow method to find

narrow_candidates = sorted(rows, key=lambda r: -r['diff_pp'])[:TOP_N]
saturated_candidates = sorted(
    [r for r in rows if r['riverside_pct'] >= SATURATION_PCT and r['rest_pct'] >= SATURATION_PCT],
    key=lambda r: -r['riverside_pct'],
)[:TOP_N]

candidates = {}
for r in narrow_candidates:
    candidates[(round(r['lon'], 4), round(r['lat'], 4))] = {**r, 'source': {'narrow'}}
for r in saturated_candidates:
    key = (round(r['lon'], 4), round(r['lat'], 4))
    if key in candidates:
        candidates[key]['source'].add('saturated')
    else:
        candidates[key] = {**r, 'source': {'saturated'}}

print(f'{len(narrow_candidates)} narrow-edge candidates + {len(saturated_candidates)} saturation '
      f'candidates -> {len(candidates)} unique cells after merge '
      f'({sum(1 for c in candidates.values() if len(c["source"]) == 2)} flagged by both).')

25 narrow-edge candidates + 25 saturation candidates -> 49 unique cells after merge (1 flagged by both).


## Pass 2 — wide-radius (1km) verification, candidates only

Same grouped-reducer pattern as notebook 05's three hand-picked hotspots, now run over every
merged candidate instead of only locations already known from the literature. Restricting this
pass to the small candidate set (not all 3016 grid cells) is what keeps it affordable — a
city-wide 1km-buffer reduceRegions pass would be far more compute than this notebook needs.

In [5]:
def wide_radius_diff(lon, lat, radius=1000):
    region = ee.Geometry.Point([lon, lat]).buffer(radius)
    grouped = builtup.rename('b').addBands(riverside_mask.rename('zone')).reduceRegion(
        reducer=ee.Reducer.mean().group(groupField=1, groupName='zone'),
        geometry=region, scale=10, maxPixels=1e9, bestEffort=True,
    ).getInfo()['groups']
    by_zone = {g['zone']: g['mean'] * 100 for g in grouped}
    return by_zone.get(1, float('nan')), by_zone.get(0, float('nan'))

results = []
for (lon, lat), r in candidates.items():
    wide_riverside, wide_rest = wide_radius_diff(lon, lat)
    results.append({
        **r,
        'lon': lon, 'lat': lat,
        'wide_riverside_pct': wide_riverside,
        'wide_rest_pct': wide_rest,
        'wide_diff_pp': wide_riverside - wide_rest,
        'source': ','.join(sorted(r['source'])),
    })

print(f'Computed wide-radius stats for {len(results)} candidate locations.')

Computed wide-radius stats for 49 candidate locations.


## Combined ranking

Rank by whichever signal is stronger for that cell (`max(diff_pp, wide_diff_pp)`) — a candidate
only needs to show a real riverside effect at *one* of the two grains to be worth surfacing;
that's the entire point of running both methods instead of picking one.

In [6]:
for r in results:
    r['best_diff_pp'] = max(r['diff_pp'], r['wide_diff_pp'])

results.sort(key=lambda r: -r['best_diff_pp'])

print(f'{"lon":>9} {"lat":>9} {"source":>16} {"narrow diff":>12} {"wide diff":>10} {"best":>8}')
for r in results[:20]:
    print(f"{r['lon']:>9.4f} {r['lat']:>9.4f} {r['source']:>16} "
          f"{r['diff_pp']:>+11.1f}pp {r['wide_diff_pp']:>+9.1f}pp {r['best_diff_pp']:>+7.1f}pp")

      lon       lat           source  narrow diff  wide diff     best
  36.7960   -1.2815           narrow       +56.4pp      -1.5pp   +56.4pp
  36.9173   -1.2906           narrow       +54.8pp      -9.1pp   +54.8pp
  36.7960   -1.2499           narrow       +52.3pp      +3.8pp   +52.3pp
  36.8005   -1.2137           narrow       +50.3pp      +8.6pp   +50.3pp
  36.9308   -1.1686           narrow       +41.5pp     +11.7pp   +41.5pp
  36.8140   -1.2770           narrow       +39.4pp     +19.0pp   +39.4pp
  37.0610   -1.2772           narrow       +38.8pp      +3.9pp   +38.8pp
  37.0745   -1.2908           narrow       +38.1pp      +2.4pp   +38.1pp
  36.8904   -1.2002           narrow       +36.5pp      -2.5pp   +36.5pp
  36.8140   -1.2725           narrow       +34.6pp     +18.6pp   +34.6pp
  36.8904   -1.2364           narrow       +34.3pp      -1.2pp   +34.3pp
  36.9757   -1.2590           narrow       +34.0pp      +5.7pp   +34.0pp
  36.9308   -1.2409           narrow       +30.7pp    

## Cross-check against known hotspots

Re-run Mathare, Kibera, and Mukuru through the combined pipeline. Expect: Mathare flagged
`narrow` (sharp local edge), Kibera flagged `saturated` and only visible via `wide_diff_pp`
(confirming the exact blind spot notebook 06 identified), Mukuru absent from both candidate lists
(no real signal at either grain, correctly excluded).

In [7]:
named_hotspots = {
    'Mathare': (36.857, -1.259),
    'Kibera': (36.789, -1.313),
    'Mukuru': (36.870, -1.310),
}

def nearest_result(lon, lat):
    return min(results, key=lambda r: (r['lon'] - lon) ** 2 + (r['lat'] - lat) ** 2)

print(f'{"Location":>10} {"source":>16} {"narrow diff":>12} {"wide diff":>10}')
for name, (lon, lat) in named_hotspots.items():
    match = nearest_result(lon, lat)
    dist_deg = ((match['lon'] - lon) ** 2 + (match['lat'] - lat) ** 2) ** 0.5
    label = match['source'] if dist_deg < 0.01 else 'not in candidate set'
    print(f"{name:>10} {label:>16} {match['diff_pp']:>+11.1f}pp {match['wide_diff_pp']:>+9.1f}pp")

  Location           source  narrow diff  wide diff
   Mathare narrow,saturated       +22.1pp      +8.6pp
    Kibera        saturated        +0.7pp      +8.5pp
    Mukuru        saturated        +0.2pp      -2.9pp


## Visualize

In [8]:
builtup_vis = {'min': 0, 'max': 1, 'palette': ['black', 'red']}
source_colors = {'narrow': 'orange', 'saturated': 'purple', 'saturated,narrow': 'red'}

Map = geemap.Map(center=[-1.290, 36.868], zoom=11)
Map.addLayer(builtup, builtup_vis, '2024 built-up')
Map.addLayer(riverside_mask.selfMask(), {'palette': ['cyan']}, 'Riparian buffer (30m)')
Map.addLayer(rivers, {'color': 'blue'}, 'River reaches (HydroSHEDS)')

for src_label, color in source_colors.items():
    pts = [r for r in results if r['source'] == src_label]
    if not pts:
        continue
    fc = ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([r['lon'], r['lat']]), {
            'diff_pp': r['diff_pp'], 'wide_diff_pp': r['wide_diff_pp'],
        }) for r in pts
    ])
    Map.addLayer(fc, {'color': color}, f'Hotspot ({src_label})')

Map.addLayerControl()
Map

Map(center=[-1.29, 36.868], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright…

In [9]:
composite_riparian = builtup.visualize(**builtup_vis).blend(
    riverside_mask.selfMask().visualize(palette=['00FFFF'], opacity=0.6)
)
url = composite_riparian.getThumbURL({'region': nairobi, 'dimensions': 900})
path = '../data/processed/nairobi_combined_hotspot_report.png'
urllib.request.urlretrieve(url, path)
print(f'Saved {path}')

Saved ../data/processed/nairobi_combined_hotspot_report.png


## Export combined hotspot table

The merged, ranked candidate table is the actual deliverable of this notebook — a single output
that carries both signals, rather than two separate notebooks a reader has to reconcile by hand.

In [10]:
import csv

out_path = '../data/processed/combined_riparian_hotspots.csv'
fieldnames = ['lon', 'lat', 'source', 'riverside_pct', 'rest_pct', 'diff_pp',
              'wide_riverside_pct', 'wide_rest_pct', 'wide_diff_pp', 'best_diff_pp']
with open(out_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results:
        writer.writerow({k: r[k] for k in fieldnames})
print(f'Saved {len(results)} ranked candidates to {out_path}')

Saved 49 ranked candidates to ../data/processed/combined_riparian_hotspots.csv


## Summary

**Candidate flagging:** of 578 grid cells with enough riverside/rest pixels to compare, the top
25 by narrow (500m within-cell) diff and top 25 by saturation (both sides >=70% built-up, sorted
by absolute riverside built-up) were merged into 49 unique candidates for the expensive 1km
wide-radius verification pass — 1 cell (Mathare's) qualified on both criteria; the rest split 24
narrow-only / 24 saturated-only.

**Top 15 combined candidates** (ranked by `best_diff_pp = max(narrow diff, wide diff)`):

| lon | lat | source | narrow diff | wide diff |
|---|---|---|---|---|
| 36.7960 | -1.2815 | narrow | +56.4pp | -1.5pp |
| 36.9173 | -1.2906 | narrow | +54.8pp | -9.1pp |
| 36.7960 | -1.2499 | narrow | +52.3pp | +3.8pp |
| 36.8005 | -1.2137 | narrow | +50.3pp | +8.6pp |
| 36.9308 | -1.1686 | narrow | +41.5pp | +11.7pp |
| 36.8140 | -1.2770 | narrow | +39.4pp | +19.0pp |
| 37.0610 | -1.2772 | narrow | +38.8pp | +3.9pp |
| 37.0745 | -1.2908 | narrow | +38.1pp | +2.4pp |
| 36.8904 | -1.2002 | narrow | +36.5pp | -2.5pp |
| 36.8140 | -1.2725 | narrow | +34.6pp | +18.6pp |
| 36.8904 | -1.2364 | narrow | +34.3pp | -1.2pp |
| 36.9757 | -1.2590 | narrow | +34.0pp | +5.7pp |
| 36.9308 | -1.2409 | narrow | +30.7pp | +2.2pp |
| 36.9757 | -1.2229 | narrow | +29.5pp | -0.2pp |
| 36.9308 | -1.2183 | narrow | +29.4pp | -16.8pp |

Every top-15-by-`best_diff_pp` slot is a `narrow` candidate — narrow diffs top out near +56pp
while wide diffs cap in the low-20s, so a pure `best_diff_pp` ranking structurally favors sharp
local edges over saturated settlements. The saturation signal is real but smaller in magnitude,
not absent: looking at `saturated`-flagged candidates ranked by `wide_diff_pp` alone surfaces a
different, still-genuine list —

| lon | lat | source | narrow diff | wide diff |
|---|---|---|---|---|
| 36.8544 | -1.2544 | saturated | +1.3pp | +14.0pp |
| 36.8589 | -1.2590 | narrow,saturated (Mathare) | +22.1pp | +8.6pp |
| 36.7870 | -1.3131 | saturated (Kibera) | +0.7pp | +8.5pp |
| 36.8589 | -1.2635 | saturated | +1.4pp | +8.0pp |
| 36.8858 | -1.2635 | saturated | +0.1pp | +7.7pp |

**This is the notebook's core result, not a side note: a single `best_diff_pp` ranking silently
buries the saturation signal under the narrow signal's larger numeric range.** A real combined
report needs to surface both ranked lists side by side (or normalize the two scores before
merging), not just take the max and sort once — recorded here as a limitation of this notebook's
current ranking, not fixed, since the raw two-signal candidate table (the CSV export) already
carries everything needed to do that properly downstream.

**Cross-check against the three known hotspots — this is where the design gets validated:**

| Location | Source flag | Narrow diff | Wide diff | Notebook 05 (1km radius) | Notebook 06 (500m grid) |
|---|---|---|---|---|---|
| Mathare | `narrow,saturated` | +22.1pp | +8.6pp | +9.6pp | +22.1pp |
| Kibera | `saturated` | +0.7pp | **+8.5pp** | +8.5pp | +0.7pp |
| Mukuru | `saturated` | +0.2pp | -2.9pp | -0.7pp | -2.7pp |

Kibera is the proof this notebook set out to get: the narrow grid method alone gives +0.7pp
(nearly invisible, matching notebook 06's original finding), but the saturation flag correctly
selects it for wide-radius verification, which recovers +8.5pp — an exact match to notebook 05's
hand-picked-hotspot number, now reached without knowing in advance that Kibera was worth checking.
Mukuru correctly stays near zero on both signals and isn't mistaken for a hotspot despite being
~100% built-up on both sides of the river — high absolute built-up alone isn't enough to flag a
false positive, because the saturation *diff* (not the absolute level) is what's compared.

**Decision: report the two ranked signals (narrow-edge, saturation/wide-radius) together, as
originally planned** — this notebook's own top-15 table shows why picking a single blended score
isn't good enough on its own; the saturation list needs to stay visible as its own ranking
alongside the narrow-edge list, not folded into one number.

**Outputs:** `data/processed/combined_riparian_hotspots.csv` (49 ranked candidates, all fields)
and `data/processed/nairobi_combined_hotspot_report.png` (map thumbnail).

**Caveats:**
- `TOP_N=25` and `SATURATION_PCT=70` are untuned analytical choices, same status as notebook 06's
  grid resolution and pixel-count filter — changing them changes which cells enter the candidate
  set before wide-radius verification ever runs.
- `best_diff_pp` ranking bias (above) means this notebook's own top-15 output underrepresents
  saturated candidates; anyone consuming the CSV directly should sort by `diff_pp` and
  `wide_diff_pp` separately rather than trusting `best_diff_pp` as a single ranking.
- No deduplication of adjacent hot cells into single sites (same caveat as notebook 06).
- Single-year 2024 snapshot — this is still not change detection (notebook 04 remains rejected
  for that); it shows where riverside built-up sits now, not whether it's worsening.
- Same river-completeness and 30m-buffer-is-analytical-not-legal caveats as notebooks 05 and 06.
